<a href="https://colab.research.google.com/github/ahmeddmanss211-create/lvl3_SecondTermprogect_ahmed_31103201702137.ipynb/blob/main/lvl3_SecondTermprogect_ahmed_31103201702137.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

gethub rebo :https://github.com/ahmeddmanss211-create/lvl3_finalprogect_ahmed_31103201702137.ipynb

# Ethics Reflection

## Why is it important to verify data collected from public APIs?
It is important to verify API data to ensure that it is accurate, complete, and reliable before using it for analysis.

## Why should data analysts document the source of their data?
Data analysts should document the data source to make the analysis transparent, traceable, and reproducible.

## How can missing or inaccurate data affect data analysis and decision-making?
Missing or inaccurate data can lead to incorrect results, misleading insights, and poor decision-making.**bold text**

In [ ]:
# Import the required libraries for data analysis, database operations, and visualization.
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt


In [ ]:
# Send a request to the GitHub API to retrieve the most-starred machine learning repositories.
import requests

url = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc&per_page=100"

response = requests.get(url)

print(response.status_code)

In [ ]:
# Convert the API response into JSON format and create a Pandas DataFrame from the repository data.
data = response.json()

df = pd.DataFrame(data["items"])



In [ ]:
# Select the important columns needed for analyzing the GitHub repositories.
df = df[
    [
        "name",
        "owner",
        "language",
        "stargazers_count",
        "forks_count",
        "watchers_count",
        "open_issues_count",
        "created_at",
        "updated_at",
        "license"
    ]
]


In [ ]:
# Extract the owner's username and license name to make the data easier to analyze.
df["owner"] = df["owner"].apply(lambda x: x["login"])
df["license"] = df["license"].apply(lambda x: x["name"] if x else None)

In [ ]:
# Display the first five rows of the dataset to inspect the collected data.
df.head()

In [ ]:
# Save the cleaned repository data into a CSV file.
df.to_csv("github_projects.csv", index=False)

In [ ]:
# Load the saved CSV file back into a DataFrame and display its first five rows.df.isnull().sum()
df = pd.read_csv("github_projects.csv")

df.head()

In [ ]:
# Check the number of missing values in each column.
df.isnull().sum()

In [ ]:
# Replace missing language and license values with "Unknown".
df["language"] = df["language"].fillna("Unknown")
df["license"] = df["license"].fillna("Unknown")

In [ ]:
# Check the dataset again to verify the remaining missing values.
df.isnull().sum()

In [ ]:
# Check whether the dataset contains any duplicate rows.
df.duplicated().sum()


In [ ]:
# Convert the repository creation and update dates into datetime format.
df["updated_at"] = pd.to_datetime(df["updated_at"])

In [ ]:
# Check the data types of the created_at and updated_at columns.
df[["created_at", "updated_at"]].dtypes


In [ ]:
# Rename selected columns to shorter and more readable names.
df = df.rename(columns={
    "stargazers_count": "stars",
    "forks_count": "forks",
    "watchers_count": "watchers",
    "open_issues_count": "open_issues",
    "created_at": "created_date",
    "updated_at": "updated_date"
})

In [ ]:
# Display the first five rows after cleaning and renaming the columns.
df.head()

In [ ]:
# Save the updated and cleaned dataset to the CSV file.
df.to_csv("github_projects.csv", index=False)

In [ ]:
# Reload the saved CSV file to verify that the data was saved correctly.
df_check = pd.read_csv("github_projects.csv")

df_check.head()

In [ ]:
# Display the number of rows and columns in the dataset.
df.shape

In [ ]:
# Load the dataset, create a SQLite database, and store the repositories in a database table.
df = pd.read_csv("github_projects.csv")

conn = sqlite3.connect("github_projects.db")

df.to_sql("Repositories", conn, if_exists="replace", index=False)

In [ ]:
# Retrieve repositories that have more than 10,000 stars.
query = """
SELECT name, owner, stars
FROM Repositories
WHERE stars > 10000
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
# Retrieve repositories whose names contain the word "Machine".
query = """
SELECT name, owner, stars
FROM Repositories
WHERE name LIKE '%Machine%'
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
# Retrieve Python repositories that have more than 10,000 stars.
query = """
SELECT name, owner, language, stars
FROM Repositories
WHERE stars > 10000
AND language = 'Python'
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
# Retrieve repositories with very high stars or forks while excluding repositories that use Python.
query = """
SELECT name, owner, language, stars, forks
FROM Repositories
WHERE (stars > 50000 OR forks > 20000)
AND NOT language = 'Python'
"""

result = pd.read_sql_query(query, conn)

result

In [ ]:
# Retrieve repositories with more than 100,000 stars or more than 30,000 forks.
query = """
SELECT name, owner, stars, forks
FROM Repositories
WHERE stars > 100000 OR forks > 30000
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Retrieve repositories that do not use Python as their programming language.
query = """
SELECT name, owner, stars, language
FROM Repositories
WHERE NOT language = 'Python'
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Retrieve the top 10 repositories ranked by the highest number of stars.
query = """
SELECT name, owner, stars
FROM Repositories
ORDER BY stars DESC
LIMIT 10
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Calculate the total number of repositories and the average number of stars.
query = """
SELECT
    COUNT(*) AS total_repositories,
    AVG(stars) AS average_stars
FROM Repositories
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Count repositories for each programming language and show languages with more than five repositories.
query = """
SELECT language, COUNT(*) AS repository_count
FROM Repositories
GROUP BY language
HAVING COUNT(*) > 5
ORDER BY repository_count DESC
"""

result = pd.read_sql_query(query, conn)
result

In [ ]:
# Retrieve the top 10 repositories by stars for further analysis.
query = """
SELECT name, stars
FROM Repositories
ORDER BY stars DESC
LIMIT 10
"""

top_repos = pd.read_sql_query(query, conn)

plt.figure(figsize=(10, 6))
plt.barh(top_repos["name"], top_repos["stars"])
plt.xlabel("Stars")
plt.ylabel("Repository")
plt.title("Top 10 Most Popular GitHub Repositories")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Retrieve repository creation dates and prepare the dates for analyzing repository creation over time.
query = """
SELECT created_date
FROM Repositories
"""

created_data = pd.read_sql_query(query, conn)

created_data["created_date"] = pd.to_datetime(created_data["created_date"])
created_data["year"] = created_data["created_date"].dt.year

yearly_repos = created_data.groupby("year").size()

plt.figure(figsize=(10, 6))
plt.plot(yearly_repos.index, yearly_repos.values, marker="o")
plt.xlabel("Year")
plt.ylabel("Number of Repositories")
plt.title("GitHub Repository Creation Trends Over Time")
plt.grid(True)
plt.show()

## Step 8: Interpretation of Results

### Key Findings

- TensorFlow is the most popular repository in the dataset with approximately 197K stars.
- The Top 10 repositories have significantly higher star counts than most other repositories.
- Python is the most common programming language, with 30 repositories.
- 23 repositories have an unknown programming language, which indicates that some language information is missing.
- The average number of stars across the 100 repositories is approximately 20,796.97.
- The repository creation trend shows how GitHub projects have grown over time.
- Overall, the analysis shows that a small number of repositories have very high popularity, while Python is the dominant programming language in the dataset.**bold text**



> Add blockquote

## Summary

The GitHub repository data was successfully stored in a SQLite database and analyzed using SQL and Python.

The analysis included filtering, searching, logical operators, sorting, limiting, aggregate functions, and grouping. The Top 10 most popular repositories were identified, and repository creation trends were visualized using Matplotlib.

The results showed that TensorFlow had the highest number of stars, while Python was the most common programming language among the repositories.***bold text***